# Этап 1. Объединение и чистка данных

**Цель:** собрать 4 файла с данными по отелям Москвы и Сочи (октябрь 2026, январь 2027) в один чистый датасет.

**Этапы работы:**
1. Загрузка 4 Excel-файлов из `../data/raw/`.
2. Объединение в один датафрейм с колонками-источниками.
3. Проверка и удаление дубликатов.
4. Приведение типов данных.
5. Анализ пропусков.
6. Проверка аномалий в ценах и датах.
7. Добавление расчётных столбцов.
8. Сохранение результата в `../data/processed/`.

In [9]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

print("Файлы в data/raw:")
print(os.listdir(RAW_PATH))

Файлы в data/raw:
['moscow_january.xlsx', 'moscow_october.xlsx', 'sochi_january.xlsx', 'sochi_october.xlsx']


## 1. Загрузка 4 файлов

Каждый файл — выгрузка из парсера по одному городу за один сезон. Добавляем к каждому две колонки-источника: `source_city` (Москва/Сочи) и `source_season` (октябрь/январь). Это нужно, чтобы отличать записи при объединении.

In [10]:
files = {
    "moscow_october.xlsx": ("Москва", "октябрь"),
    "moscow_january.xlsx": ("Москва", "январь"),
    "sochi_october.xlsx":  ("Сочи", "октябрь"),
    "sochi_january.xlsx":  ("Сочи", "январь"),
}

frames = []

for filename, (city, season) in files.items():
    filepath = f"{RAW_PATH}/{filename}"
    df_temp = pd.read_excel(filepath, sheet_name="Data")
    df_temp["source_city"] = city
    df_temp["source_season"] = season
    df_temp["source_file"] = filename
    frames.append(df_temp)
    print(f"{filename}: {df_temp.shape[0]} строк, {df_temp.shape[1]} столбцов")

print(f"Всего загружено файлов: {len(frames)}")

moscow_october.xlsx: 260 строк, 65 столбцов
moscow_january.xlsx: 350 строк, 65 столбцов
sochi_october.xlsx: 250 строк, 62 столбцов
sochi_january.xlsx: 250 строк, 62 столбцов
Всего загружено файлов: 4


## 2. Объединение в один датафрейм

Склеиваем 4 датафрейма через `pd.concat`. `ignore_index=True` перенумеровывает строки с нуля.

In [11]:
df = pd.concat(frames, ignore_index=True)

print(f"Объединённый датафрейм: {df.shape[0]} строк, {df.shape[1]} столбцов")
print(f"Распределение по городам и сезонам:")
print(df.groupby(["source_city", "source_season"]).size())

Объединённый датафрейм: 1110 строк, 65 столбцов
Распределение по городам и сезонам:
source_city  source_season
Москва       октябрь          260
             январь           350
Сочи         октябрь          250
             январь           250
dtype: int64


## 3. Первичный анализ

In [12]:
df.head(10)

,address,checkIn,checkOut,city,country,currency,distanceFromCenterKm,freeCancellation,hasBreakfast,hasSecureDeposit,hotelId,images/0,images/1,images/2,images/3,images/4,images/5,images/6,images/7,images/8,images/9,isGuruHotel,isSuperhost,kind,latitude,longitude,mainImage,masterId,metroNearby,minPrice,minPricePerNight,name,nameEn,nearestSubway,nearestSubwayDistanceM,otaChainId,priceEur,priceRub,priceUsd,rates/0/currency,rates/0/freeCancellation,rates/0/freeCancellationBefore,rates/0/hasBreakfast,rates/0/mealType,rates/0/price,rates/0/priceEur,rates/0/pricePerNight,rates/0/priceRub,rates/0/priceUsd,rates/0/roomName,ratesCount,rating,ratingCount,regionId,regionName,roomsNumber,scrapedAt,starRating,tripadvisorRating,tripadvisorReviews,type,url,source_city,source_season,source_file
0,"Samarskaya street, 1, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,3.249540,True,False,False,radisson_blu_olympiyskiy_hotel,https://cdn.ostrovok.ru/t/1024x768/extranet/2b...,https://cdn.ostrovok.ru/t/1024x768/extranet/58...,https://cdn.ostrovok.ru/t/1024x768/extranet/9a...,https://cdn.ostrovok.ru/t/1024x768/extranet/60...,https://cdn.ostrovok.ru/t/1024x768/extranet/b5...,https://cdn.ostrovok.ru/t/1024x768/extranet/b5...,https://cdn.ostrovok.ru/t/1024x768/extranet/99...,https://cdn.ostrovok.ru/t/1024x768/extranet/78...,https://cdn.ostrovok.ru/t/1024x768/extranet/de...,https://cdn.ostrovok.ru/t/1024x768/extranet/0a...,True,False,Hotel,55.784706,37.626274,https://cdn.ostrovok.ru/t/1024x768/extranet/2b...,8869780,True,143000.0,14300,"Гостиница Radisson Blu Олимпийский, Москва","Radisson Blu Olympiyskiy Hotel, Moscow",Проспект Мира (Калужско-Рижская),548.0,18.0,1491.83,143000.0,1700.86,RUB,True,2026-10-14T21:00:00Z,False,nomeal,143000.0,1491.83,14300,143000.0,1700.86,Двухместный номер Стандартный с видом на город...,1,9.1,728.0,2395,Москва,379.0,2026-09-27T07:36:50.471Z,5.0,NaN,NaN,awarded-hotel,https://ostrovok.ru/hotel/russia/moscow/mid886...,Москва,октябрь,moscow_october.xlsx
1,"150 Mira Prospect, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,7.581613,True,False,False,cosmos_hotel,https://cdn.ostrovok.ru/t/1024x768/extranet/4a...,https://cdn.ostrovok.ru/t/1024x768/extranet/07...,https://cdn.ostrovok.ru/t/1024x768/extranet/9f...,https://cdn.ostrovok.ru/t/1024x768/extranet/f5...,https://cdn.ostrovok.ru/t/1024x768/extranet/b9...,https://cdn.ostrovok.ru/t/1024x768/extranet/4e...,https://cdn.ostrovok.ru/t/1024x768/extranet/3e...,https://cdn.ostrovok.ru/t/1024x768/extranet/b4...,https://cdn.ostrovok.ru/t/1024x768/extranet/f2...,https://cdn.ostrovok.ru/t/1024x768/extranet/46...,True,False,Hotel,55.822060,37.647100,https://cdn.ostrovok.ru/t/1024x768/extranet/4a...,7466678,True,46200.0,4620,Cosmos Москва ВДНХ Отель,Cosmos Moscow VDNH Hotel,ВДНХ,384.0,873.0,481.97,46200.0,549.51,RUB,True,2026-10-14T21:00:00Z,False,nomeal,46200.0,481.97,4620,46200.0,549.51,Номер Стандарт,1,6.9,2533.0,2395,Москва,1644.0,2026-09-27T07:36:50.471Z,3.0,NaN,NaN,historic-hotel,https://ostrovok.ru/hotel/russia/moscow/mid746...,Москва,октябрь,moscow_october.xlsx
2,"Ленинский проспект, д.158, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,14.356438,True,False,False,happy_home_apartment_hotel,https://cdn.ostrovok.ru/t/1024x768/extranet/81...,https://cdn.ostrovok.ru/t/1024x768/extranet/cd...,https://cdn.ostrovok.ru/t/1024x768/extranet/70...,https://cdn.ostrovok.ru/t/1024x768/extranet/67...,https://cdn.ostrovok.ru/t/1024x768/extranet/a1...,https://cdn.ostrovok.ru/t/1024x768/extranet/d2...,https://cdn.ostrovok.ru/t/1024x768/extranet/d7...,https://cdn.ostrovok.ru/t/1024x768/extranet/56...,https://cdn.ostrovok.ru/t/1024x768/extranet/7e...,https://cdn.ostrovok.ru/t/1024x768/extranet/48...,True,False,Hotel,55.651367,37.482860,https://cdn.ostrovok.ru/t/1024x768/extranet/81...,13647556,True,45600.0,4560,Апарт-отель Happy Home,Happy Home Apartment hotel,Тропарёво,892.0,NaN,475.72,45600.0,542.37,RUB,True,2026-10-14T12:00:00Z,False,nomeal,45600.0,475.72,4560,45600.0,542.37,Апартаменты Comfort c 1 

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 65 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   address                         1110 non-null   str    
 1   checkIn                         1110 non-null   str    
 2   checkOut                        1110 non-null   str    
 3   city                            1110 non-null   str    
 4   country                         1110 non-null   str    
 5   currency                        1110 non-null   str    
 6   distanceFromCenterKm            1110 non-null   float64
 7   freeCancellation                1110 non-null   bool   
 8   hasBreakfast                    1110 non-null   bool   
 9   hasSecureDeposit                1110 non-null   bool   
 10  hotelId                         1110 non-null   str    
 11  images/0                        1110 non-null   str    
 12  images/1                        1110 non-null

## 4. Полный список столбцов

В датафрейме 65 столбцов. Ключевые группы:
- **Идентификация:** `hotelId`, `masterId`, `name`, `url`
- **Локация:** `city`, `address`, `latitude`, `longitude`, `distanceFromCenterKm`, `nearestSubway`
- **Цены:** `priceRub`, `priceUsd`, `priceEur`, `minPricePerNight`
- **Рейтинг:** `rating`, `ratingCount`, `starRating`, `tripadvisorRating`
- **Условия:** `freeCancellation`, `hasBreakfast`
- **Тарифы:** `rates/0/price`, `rates/0/roomName`
- **Служебные:** `source_city`, `source_season`, `source_file`

Что важно: `checkIn`, `checkOut`, `scrapedAt` ещё хранятся как текст (`object`). `metroNearby` - тоже `object` из-за пропусков, а не `bool`. Приведём их к правильным типам

In [14]:
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

1. address
2. checkIn
3. checkOut
4. city
5. country
6. currency
7. distanceFromCenterKm
8. freeCancellation
9. hasBreakfast
10. hasSecureDeposit
11. hotelId
12. images/0
13. images/1
14. images/2
15. images/3
16. images/4
17. images/5
18. images/6
19. images/7
20. images/8
21. images/9
22. isGuruHotel
23. isSuperhost
24. kind
25. latitude
26. longitude
27. mainImage
28. masterId
29. metroNearby
30. minPrice
31. minPricePerNight
32. name
33. nameEn
34. nearestSubway
35. nearestSubwayDistanceM
36. otaChainId
37. priceEur
38. priceRub
39. priceUsd
40. rates/0/currency
41. rates/0/freeCancellation
42. rates/0/freeCancellationBefore
43. rates/0/hasBreakfast
44. rates/0/mealType
45. rates/0/price
46. rates/0/priceEur
47. rates/0/pricePerNight
48. rates/0/priceRub
49. rates/0/priceUsd
50. rates/0/roomName
51. ratesCount
52. rating
53. ratingCount
54. regionId
55. regionName
56. roomsNumber
57. scrapedAt
58. starRating
59. tripadvisorRating
60. tripadvisorReviews
61. type
62. url
63. source_c

## 5. Проверка дубликатов

Один и тот же отель может встретиться два раза - если парсер встретил его на двух страницах выдачи. Проверяем по комбинации `hotelId`,`checkIn`,`checkOut` -  если эти три поля совпадают, значит, это одна и та же запись

In [15]:
dup_mask = df.duplicated(subset=["hotelId", "checkIn", "checkOut"], keep=False)
print(f"Строк в дубликатах: {dup_mask.sum()}")

if dup_mask.sum() > 0:
    dup_examples = df[dup_mask].sort_values(["hotelId", "checkIn"]).head(10)
    print(dup_examples[["hotelId", "name", "checkIn", "checkOut", "priceRub", "source_file"]])

Строк в дубликатах: 0


## 6. Удаление дубликатов

Дубликатов нет - парсер не пересекал отели между страницами.

In [16]:
df_clean = df.drop_duplicates(
    subset=["hotelId", "checkIn", "checkOut"],
    keep="first"
).copy()

print(f"Было: {df.shape[0]} строк")
print(f"Стало: {df_clean.shape[0]} строк")
print(f"Удалено: {df.shape[0] - df_clean.shape[0]} строк")

Было: 1110 строк
Стало: 1110 строк
Удалено: 0 строк


## 7. Приведение типов

Большинство числовых полей уже в правильных типах. Необходимо привести:
- `checkIn`, `checkOut`, `scrapedAt` - к `datetime64`.
- `metroNearby` - к `bool` (сейчас `object` из-за пропусков).

In [17]:
date_cols = ["checkIn", "checkOut", "scrapedAt"]
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

df_clean["metroNearby"] = df_clean["metroNearby"].astype("boolean")

print("Типы полей после приведения")
print(df_clean[["checkIn", "checkOut", "scrapedAt", "metroNearby",
                "priceRub", "rating", "starRating", "distanceFromCenterKm"]].dtypes)

Типы полей после приведения
checkIn                      datetime64[us]
checkOut                     datetime64[us]
scrapedAt               datetime64[us, UTC]
metroNearby                         boolean
priceRub                            float64
rating                              float64
starRating                          float64
distanceFromCenterKm                float64
dtype: object


## 8. Анализ пропусков

В каких столбцах есть пропуски.
- **100% заполнены:** цены, рейтинг, координаты, идентификаторы.
- **Частично пустые:** `starRating`, `tripadvisorRating`, `metroNearby`, `otaChainId`, `type`, `roomsNumber`.
- **Картинки** (`images/3` — `images/9`) - у каждого отеля своё число фото.

Это нормально, так как не все отели имеют звёзды, Tripadvisor или метро рядом.

In [18]:
missing = df_clean.isnull().sum()
missing_pct = (missing/len(df_clean) * 100).round(2)

missing_df = pd.DataFrame({
    "Пропусков": missing,
    "Процент": missing_pct
}).sort_values("Процент", ascending=False)

missing_df[missing_df["Пропусков"] > 0]

,Пропусков,Процент
otaChainId,935,84.23
type,832,74.95
tripadvisorReviews,651,58.65
tripadvisorRating,651,58.65
metroNearby,551,49.64
nearestSubway,519,46.76
nearestSubwayDistanceM,519,46.76
starRating,507,45.68
roomsNumber,203,18.29
rates/0/freeCancellationBefore,160,14.41


In [19]:
no_price = df_clean["priceRub"].isnull().sum()
no_rating = df_clean["rating"].isnull().sum()
no_dist = df_clean["distanceFromCenterKm"].isnull().sum()

print(f"Отелей без цены: {no_price} ({no_price/len(df_clean)*100:.1f}%)")
print(f"Отелей без рейтинга: {no_rating} ({no_rating/len(df_clean)*100:.1f}%)")
print(f"Отелей без расстояния до центра: {no_dist} ({no_dist/len(df_clean)*100:.1f}%)")

Отелей без цены: 0 (0.0%)
Отелей без рейтинга: 0 (0.0%)
Отелей без расстояния до центра: 0 (0.0%)


## 9. Удаление строк без цены

Проверяем на всякий случай. Если отелей без цены < 5%, тогдаа удаляем, если больше - оставляем для разбора. По `df.info()` в `priceRub` пропусков нет, поэтому ничего не удаляем.

In [20]:
if no_price / len(df_clean) < 0.05:
    df_clean = df_clean.dropna(subset=["priceRub"]).copy()
    print(f"Удалено {no_price} строк без цены. Осталось: {len(df_clean)}")
else:
    print(f"Без цены {no_price} отелей — слишком много, не удаляем.")

Удалено 0 строк без цены. Осталось: 1110


## 10. Проверка аномалий в ценах

`priceRub` — цена за весь период. `minPricePerNight` — за одну ночь.

Смотрим: нет ли нулевых, отрицательных или аномально высоких цен. Топ-5 дорогих и дешёвых — чтобы понять, что в выборке есть и люксы, и хостелы.

In [21]:
print("Статистика по priceRub:")
print(df_clean["priceRub"].describe())

print("Самые дорогие:")
print(df_clean.nlargest(5, "priceRub")[["name", "source_city", "source_season", "priceRub", "minPricePerNight"]])

print("Самые дешёвые:")
print(df_clean.nsmallest(5, "priceRub")[["name", "source_city", "source_season", "priceRub", "minPricePerNight"]])

Статистика по priceRub:
count      1110.000000
mean      64181.330108
std       54582.040798
min        8000.000000
25%       33132.250000
50%       51425.000000
75%       75527.750000
max      777777.000000
Name: priceRub, dtype: float64
Самые дорогие:
                                      name source_city source_season  priceRub  minPricePerNight
524                      Отель Шереметьево      Москва        январь  777777.0            111111
149                     Лотте Отель Москва      Москва       октябрь  485000.0             48500
219               Арарат Парк Хаятт Москва      Москва       октябрь  399500.0             39950
508                        Отель Метрополь      Москва        январь  385000.0             55000
177  Отель The St. Regis Moscow Nikolskaya      Москва       октябрь  357000.0             35700
Самые дешёвые:
                                 name source_city source_season  priceRub  minPricePerNight
814     Меблированные комнаты Эверест        Сочи       о

## 11. Проверка диапазона дат

В данных должны быть только запрошенные даты:

Октябрь: 15.10.2026 - 25.10.2026
Январь: 02.01.2027 - 09.01.2027

In [22]:
print("Уникальные checkIn:")
print(sorted(df_clean["checkIn"].unique()))

print("Уникальные checkOut:")
print(sorted(df_clean["checkOut"].unique()))

print("Распределение по датам, городам и сезонам:")
print(df_clean.groupby(["source_city", "source_season", "checkIn", "checkOut"]).size())

Уникальные checkIn:
[Timestamp('2026-10-15 00:00:00'), Timestamp('2027-01-02 00:00:00')]
Уникальные checkOut:
[Timestamp('2026-10-25 00:00:00'), Timestamp('2027-01-09 00:00:00')]
Распределение по датам, городам и сезонам:
source_city  source_season  checkIn     checkOut  
Москва       октябрь        2026-10-15  2026-10-25    260
             январь         2027-01-02  2027-01-09    350
Сочи         октябрь        2026-10-15  2026-10-25    250
             январь         2027-01-02  2027-01-09    250
dtype: int64


## 12. Добавление расчётных столбцов
Добавляем поля для дальнейшего анализа:
- `nights` — количество ночей (10 для октября, 7 для января).
- `price_per_night_calc` — цена за ночь, посчитанная как `priceRub / nights`. Используем для сравнения сезонов с разной длиной поездки.
- `distance_category` — категория расстояния от центра (0–2, 2–5, 5–10, 10–20, 20+ км).
- `rating_category` — категория рейтинга (<7, 7–8, 8–9, 9+).

In [23]:
df_clean["nights"] = (df_clean["checkOut"] - df_clean["checkIn"]).dt.days
df_clean["price_per_night_calc"] = df_clean["priceRub"] / df_clean["nights"]

bins = [0, 2, 5, 10, 20, 100]
labels = ["0-2 км", "2-5 км", "5-10 км", "10-20 км", "20+ км"]
df_clean["distance_category"] = pd.cut(
    df_clean["distanceFromCenterKm"], bins=bins, labels=labels
)

df_clean["rating_category"] = pd.cut(
    df_clean["rating"], bins=[0, 7, 8, 9, 10], labels=["<7", "7-8", "8-9", "9+"]
)

print(df_clean[["nights", "price_per_night_calc", "distance_category", "rating_category"]].head())

   nights  price_per_night_calc distance_category rating_category
0      10               14300.0            2-5 км              9+
1      10                4620.0           5-10 км              <7
2      10                4560.0          10-20 км             7-8
3      10                3213.0            2-5 км             7-8
4      10                3618.0           5-10 км             7-8


In [24]:
# Считаем количество непустых фото у каждого отеля
image_cols = [f"images/{i}" for i in range(10)]
df_clean["photo_count"] = df_clean[image_cols].notna().sum(axis=1)

print("Распределение по числу фото:")
print(df_clean["photo_count"].value_counts().sort_index())

print(f"\nСреднее число фото: {df_clean['photo_count'].mean():.1f}")

# Удаляем ссылки на фото и url отеля
cols_to_drop = image_cols + ["mainImage", "url"]
df_clean = df_clean.drop(columns=cols_to_drop)

print(f"Удалено : {len(cols_to_drop)}")
print(f"Колонок стало: {df_clean.shape[1]}")

Распределение по числу фото:
photo_count
3      10
4       5
5      11
6      38
7      18
8      20
9      23
10    985
Name: count, dtype: int64

Среднее число фото: 9.6
Удалено : 12
Колонок стало: 58


## 13. Финальная проверка

In [25]:
print("ИТОГОВЫЙ ДАТАСЕТ")
print(f"Строк: {len(df_clean)}")
print(f"Столбцов: {df_clean.shape[1]}")
print(f"Города: {df_clean['source_city'].unique().tolist()}")
print(f"Сезоны: {df_clean['source_season'].unique().tolist()}")
print(f"Распределение:")
print(df_clean.groupby(["source_city", "source_season"]).size())
print(f"Пропуски в ключевых полях:")
for col in ["priceRub", "rating", "starRating", "distanceFromCenterKm"]:
    n = df_clean[col].isnull().sum()
    print(f" {col}: {n} ({n/len(df_clean)*100:.1f}%)")

ИТОГОВЫЙ ДАТАСЕТ
Строк: 1110
Столбцов: 58
Города: ['Москва', 'Сочи']
Сезоны: ['октябрь', 'январь']
Распределение:
source_city  source_season
Москва       октябрь          260
             январь           350
Сочи         октябрь          250
             январь           250
dtype: int64
Пропуски в ключевых полях:
 priceRub: 0 (0.0%)
 rating: 0 (0.0%)
 starRating: 507 (45.7%)
 distanceFromCenterKm: 0 (0.0%)


## 14. Сохранение результатов
Сохраняем три файла в `../data/processed/`:
- `hotels_all.csv` - полный датасет для анализа.
- `hotels_all.xlsx` - для быстрого просмотра в Excel.
- `sample.csv` - первые 100 строк для GitHub.

In [26]:
for col in ["scrapedAt", "checkIn", "checkOut"]:
    if df_clean[col].dt.tz is not None:
        df_clean[col] = df_clean[col].dt.tz_localize(None)

print("Таймзона убрана. Проверка:")
print(df_clean[["scrapedAt", "checkIn", "checkOut"]].dtypes)

Таймзона убрана. Проверка:
scrapedAt    datetime64[us]
checkIn      datetime64[us]
checkOut     datetime64[us]
dtype: object


In [27]:
df_clean.to_csv(f"{PROCESSED_PATH}/hotels_all.csv", index=False, encoding="utf-8-sig")
print(f"Сохранено: {PROCESSED_PATH}/hotels_all.csv")

df_clean.to_excel(f"{PROCESSED_PATH}/hotels_all.xlsx", index=False)
print(f"Сохранено: {PROCESSED_PATH}/hotels_all.xlsx")

df_clean.head(100).to_csv(f"{PROCESSED_PATH}/sample.csv", index=False, encoding="utf-8-sig")
print(f"Сохранено: {PROCESSED_PATH}/sample.csv")

print(f"Размер датасета: {df_clean.shape}")

Сохранено: ../data/processed/hotels_all.csv
Сохранено: ../data/processed/hotels_all.xlsx
Сохранено: ../data/processed/sample.csv
Размер датасета: (1110, 58)


In [28]:
df_clean.head(10)

,address,checkIn,checkOut,city,country,currency,distanceFromCenterKm,freeCancellation,hasBreakfast,hasSecureDeposit,hotelId,isGuruHotel,isSuperhost,kind,latitude,longitude,masterId,metroNearby,minPrice,minPricePerNight,name,nameEn,nearestSubway,nearestSubwayDistanceM,otaChainId,priceEur,priceRub,priceUsd,rates/0/currency,rates/0/freeCancellation,rates/0/freeCancellationBefore,rates/0/hasBreakfast,rates/0/mealType,rates/0/price,rates/0/priceEur,rates/0/pricePerNight,rates/0/priceRub,rates/0/priceUsd,rates/0/roomName,ratesCount,rating,ratingCount,regionId,regionName,roomsNumber,scrapedAt,starRating,tripadvisorRating,tripadvisorReviews,type,source_city,source_season,source_file,nights,price_per_night_calc,distance_category,rating_category,photo_count
0,"Samarskaya street, 1, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,3.249540,True,False,False,radisson_blu_olympiyskiy_hotel,True,False,Hotel,55.784706,37.626274,8869780,True,143000.0,14300,"Гостиница Radisson Blu Олимпийский, Москва","Radisson Blu Olympiyskiy Hotel, Moscow",Проспект Мира (Калужско-Рижская),548.0,18.0,1491.83,143000.0,1700.86,RUB,True,2026-10-14T21:00:00Z,False,nomeal,143000.0,1491.83,14300,143000.0,1700.86,Двухместный номер Стандартный с видом на город...,1,9.1,728.0,2395,Москва,379.0,2026-09-27 07:36:50.471,5.0,NaN,NaN,awarded-hotel,Москва,октябрь,moscow_october.xlsx,10,14300.00,2-5 км,9+,10
1,"150 Mira Prospect, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,7.581613,True,False,False,cosmos_hotel,True,False,Hotel,55.822060,37.647100,7466678,True,46200.0,4620,Cosmos Москва ВДНХ Отель,Cosmos Moscow VDNH Hotel,ВДНХ,384.0,873.0,481.97,46200.0,549.51,RUB,True,2026-10-14T21:00:00Z,False,nomeal,46200.0,481.97,4620,46200.0,549.51,Номер Стандарт,1,6.9,2533.0,2395,Москва,1644.0,2026-09-27 07:36:50.471,3.0,NaN,NaN,historic-hotel,Москва,октябрь,moscow_october.xlsx,10,4620.00,5-10 км,<7,10
2,"Ленинский проспект, д.158, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,14.356438,True,False,False,happy_home_apartment_hotel,True,False,Hotel,55.651367,37.482860,13647556,True,45600.0,4560,Апарт-отель Happy Home,Happy Home Apartment hotel,Тропарёво,892.0,NaN,475.72,45600.0,542.37,RUB,True,2026-10-14T12:00:00Z,False,nomeal,45600.0,475.72,4560,45600.0,542.37,Апартаменты Comfort c 1 комнатой (двуспальная ...,1,7.9,30.0,2395,Москва,25.0,2026-09-27 07:36:50.471,NaN,NaN,NaN,NaN,Москва,октябрь,moscow_october.xlsx,10,4560.00,10-20 км,7-8,10
3,"переулок Съезжинский, д.8 стр.1, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,3.592184,True,False,False,otel_andron,True,False,Hotel,55.747700,37.673280,6344401,True,32130.0,3213,Гостиница Андрон-Отель,Andron-Otel Hotel,Площадь Ильича,465.0,NaN,335.19,32130.0,382.16,RUB,True,2026-10-15T11:00:00Z,False,nomeal,32130.0,335.19,3213,32130.0,382.16,Двухместный номер Economy (без окна) (двуспаль...,1,7.3,847.0,2395,Москва,94.0,2026-09-27 07:36:50.471,3.0,2.7,130.0,NaN,Москва,октябрь,moscow_october.xlsx,10,3213.00,2-5 км,7-8,10
4,"улица Петра Романова, д.7/2, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,6.645786,True,False,False,aroma_hotel_on_kozhukhovskaya,True,False,Hotel,55.708160,37.681767,9084838,True,36180.0,3618,Отель Арома-Отель,Aroma-otel Hotel,Кожуховская,322.0,NaN,377.44,36180.0,430.33,RUB,True,2026-10-15T11:00:00Z,False,nomeal,36180.0,377.44,3618,36180.0,430.33,Двухместный номер Budget Цокольный этаж (двусп...,1,8.0,631.0,2395,Москва,124.0,2026-09-27 07:36:50.472,3.0,NaN,NaN,NaN,Москва,октябрь,moscow_october.xlsx,10,3618.00,5-10 км,7-8,10
5,"улица Большая Семеновская, д. 32, стр. 3, Москва",2026-10-15,2026-10-25,Москва,Россия,RUB,6.306223,True,False,False,shelterz_elektrozavodskaya,True,False,Hotel,55.781400,37.707794,9990836,True,29767.5,2977,Отель Shelterz Электрозаводская,Shelterz Elektrozavodskaya Hotel,Электрозаводская (Арбатско-Покровская),175.0,NaN,310.55,29767.5,354.06,RUB,True,2026-10-14T11:00:00Z,False,nomeal,29767.5,310.55,2977,29767.5,354.06,Двухместный номер Стандартный без окна (общая ...,1,8.9,903.0,2395,Москва,90.0,2026-

<class 'pandas.Series'>
RangeIndex: 1110 entries, 0 to 1109
Series name: scrapedAt
Non-Null Count  Dtype         
--------------  -----         
1110 non-null   datetime64[us]
dtypes: datetime64[us](1)
memory usage: 8.8 KB
